# Building the GPT Transformer Block

Welcome to the heart of the machine! In the last notebook, we defined a configuration for our model. Now, we're going to build the engine itself. By the end of this notebook, you will understand the fundamental building block of a GPT model, the Transformer Block. You will assemble its key components—Causal Self-Attention and a simple neural network—from scratch. Finally, you will see how these blocks are stacked to create the complete GPT architecture, ready to learn from text.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
from dataclasses import dataclass

# We'll re-use the GPTConfig from the previous notebook
# It's a simple container for the model's hyperparameters
@dataclass
class GPTConfig:
    block_size: int = 256
    vocab_size: int = 65 # GPT-2 vocab_size is 50257, 65 is for our simple shakespeare dataset
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    dropout: float = 0.2
    bias: bool = False # We're not using bias in our layers

### The Core Idea: A Committee Meeting

Imagine you're trying to write an essay, one word at a time. The Transformer architecture works in a surprisingly similar way, but instead of just looking at the single last word you wrote, it can look at *all* the previous words at once to decide on the next one.

The "Transformer Block" is the core component that enables this. Think of it as a sophisticated "thinking step" that refines the meaning of each word in your sequence. We can use an analogy: a committee meeting.

Each word in your sentence is a person at a meeting. A single Transformer Block is one round of discussion and thinking.

1.  **Causal Self-Attention (The Discussion):** This is where every person (token) communicates with every *other person who has already spoken*. A token looks at the previous tokens and asks, "Given my own identity, which of you are most relevant to what I'm trying to say next?" It then creates a new, context-rich summary of itself based on this discussion. The "causal" part is crucial: a person can't listen to someone who hasn't spoken yet. This ensures our model can't cheat by looking at future words to predict the current one.

2.  **MLP / Feed-Forward Network (The Thinking):** After the group discussion, each person (token) goes off to their own office to think individually. They take the new context-rich summary they just formed and process it through their own private neural network. This allows for more complex, non-linear ideas to be developed from the information gathered during the discussion.

3.  **Residual Connections (The Memory):** After both the "discussion" and "thinking" phases, the model adds the *original* input to the output of the phase. This is like saying, "Let's incorporate these new insights, but without forgetting our original point." It's a simple but powerful trick that helps the model learn much more effectively.

A full GPT model simply stacks these "committee meetings" (Transformer Blocks) on top of each other. The output of the first block becomes the input to the second, allowing the model to develop progressively more sophisticated interpretations of the text.

In [ ]:
class CausalSelfAttention(nn.Module):
    """ The "discussion" part of our committee meeting """

    def __init__(self, config):
        super().__init__()
        # We need n_embd to be divisible by n_head
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        
        # A single linear layer to produce query, key, and value matrices
        # This is more efficient than three separate linear layers
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # The output projection layer
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        
        # A buffer for the causal mask. 'register_buffer' makes it part of the
        # model's state but not a parameter to be trained.
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # Batch size, Sequence length, Embedding dimensionality

        # 1. Get query, key, and value from the input
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        
        # 2. Reshape for multi-head attention
        # (B, T, C) -> (B, n_head, T, head_size)
        head_size = C // self.n_head
        k = k.view(B, T, self.n_head, head_size).transpose(1, 2)
        q = q.view(B, T, self.n_head, head_size).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_size).transpose(1, 2)
        
        # 3. Calculate attention scores ("affinities")
        # (B, nh, T, hs) @ (B, nh, hs, T) -> (B, nh, T, T)
        wei = (q @ k.transpose(-2, -1)) * (head_size**-0.5)
        
        # 4. Apply the causal mask
        wei = wei.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        
        # 5. Apply attention to values
        # (B, nh, T, T) @ (B, nh, T, hs) -> (B, nh, T, hs)
        out = wei @ v
        
        # 6. Reshape back to the original format and project
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.c_proj(out)
        return out

class MLP(nn.Module):
    """ The "thinking" part of our committee meeting """

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):
    """ A single Transformer Block: one round of discussion and thinking """

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        # The residual connections are the '+' operations
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

### Walking Through the Code

Let's break down the `Block` and its components, connecting them back to our analogy.

**`CausalSelfAttention` (The Discussion)**

*   `self.c_attn = nn.Linear(...)`: This is where we generate the three key vectors for each token:
    *   **Query (Q):** What I'm looking for.
    *   **Key (K):** What I contain.
    *   **Value (V):** What I will communicate if you pay attention to me.
*   `.split()` and `.view().transpose()`: This is the logic for "multi-head attention." Instead of having one big discussion, we split the committee into smaller sub-committees (`n_head`). Each sub-committee focuses on a different aspect of the text. This is more effective than one single discussion. The code reshapes the `(Batch, Time, Channels)` tensor into `(Batch, NumHeads, Time, HeadSize)`.
*   `wei = (q @ k.transpose(...))`: This is the core of attention! We perform a matrix multiplication between every token's Query and every other token's Key. The result is an "affinity matrix" or a "score" of how much each token should pay attention to every other token. We scale it by `head_size**-0.5` to keep the values stable.
*   `wei = wei.masked_fill(...)`: Here's the "causal" part. We use a triangular mask (`self.bias`) to set all the scores for "future" tokens to negative infinity. When we apply `softmax`, these scores will become zero, ensuring a token can't get information from tokens that come after it.
*   `out = wei @ v`: We use the attention scores (`wei`) to create a weighted sum of the Value vectors. Tokens that scored highly will contribute more to the output for the current token.
*   `out = self.c_proj(out)`: Finally, we combine the results from all the sub-committees (heads) back into a single vector and run it through a final linear layer.

**`MLP` (The Thinking)**

This module is more straightforward. It's a standard two-layer feed-forward network.
*   `self.c_fc`: The first layer expands the embedding dimension by a factor of 4. This gives the model more "room to think."
*   `self.gelu`: A non-linear activation function. This is where the complex, non-linear processing happens.
*   `self.c_proj`: The second layer projects the result back down to the original embedding dimension.

**`Block` (Putting it all together)**

The `Block` class itself is very simple. It just arranges the pieces in the right order. Notice the flow in the `forward` method:
1.  `self.ln_1(x)`: Apply Layer Normalization. This stabilizes the network.
2.  `self.attn(...)`: Pass the result to the attention mechanism.
3.  `x = x + ...`: Add the output of the attention back to the original input (`x`). This is the first **residual connection**.
4.  `self.ln_2(x)`: Another Layer Normalization.
5.  `self.mlp(...)`: Pass the result to the MLP for "thinking."
6.  `x = x + ...`: Add the output of the MLP back to its input. This is the second **residual connection**.

This `x + ...` pattern is what makes deep networks trainable. It ensures that we never lose the original signal as we pass through many layers of computation.

In [ ]:
# Create a sample config
config = GPTConfig(
    block_size=8,
    vocab_size=10,
    n_layer=4,
    n_head=4,
    n_embd=32,
)

# Instantiate a single Transformer Block
transformer_block = Block(config)
print("Transformer Block instantiated successfully.")

# Create a dummy input tensor
# Batch size = 2, Sequence length = 5, Embedding dim = 32
batch_size = 2
seq_len = 5
input_tensor = torch.randn(batch_size, seq_len, config.n_embd)

# Pass the tensor through the block
output_tensor = transformer_block(input_tensor)

# Print shapes to verify
print(f"Input shape:  {input_tensor.shape}")
print(f"Output shape: {output_tensor.shape}")

# The output shape is identical to the input shape, as expected.
# The block refines the information at each position but doesn't change the tensor's dimensions.
assert input_tensor.shape == output_tensor.shape

### Going Deeper: Assembling the Full GPT Model

A single block performs one step of refinement. The real power of GPT comes from stacking these blocks sequentially. The output of the first block becomes the input to the second, and so on. This allows the model to build up representations of the text that are increasingly abstract and context-aware.

Let's build the full `GPT` class. It's primarily a container for the blocks and the other necessary layers:

*   **Token Embeddings (`wte`):** A lookup table to convert token indices (like `[5, 23, 1]`) into dense vectors.
*   **Positional Embeddings (`wpe`):** A lookup table that provides a unique vector for each position in the sequence (`[0, 1, 2, ...]`). This is crucial because, unlike RNNs, the attention mechanism itself has no inherent sense of order. We add these to the token embeddings to give the model that information.
*   **A list of Transformer Blocks (`h`):** This is the core of the model, where we stack `n_layer` blocks.
*   **A final Layer Norm (`ln_f`):** One final normalization step after the blocks.
*   **A Language Model Head (`lm_head`):** A final linear layer that maps the refined token embeddings back to the size of our vocabulary. The output values (logits) represent the model's prediction for the next token in the sequence.

In [ ]:
class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.config = config

        # The model is a dictionary of modules
        self.transformer = nn.ModuleDict(dict(
            # Token and positional embeddings
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            # The stack of Transformer Blocks
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            # Final layer norm
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        # The final linear layer to map to vocabulary
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.size()
        
        # 1. Get token and position embeddings
        tok_emb = self.transformer.wte(idx) # (B, T, n_embd)
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        pos_emb = self.transformer.wpe(pos) # (T, n_embd)
        
        # 2. Add them together
        x = tok_emb + pos_emb
        
        # 3. Pass through the stack of blocks
        for block in self.transformer.h:
            x = block(x)
            
        # 4. Final normalization and language model head
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        
        return logits

# --- Let's test the full model ---

# Use the same config as before
model = GPT(config)
print(f"Model instantiated with {config.n_layer} layers.")
print(f"Total parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")


# Create a dummy input tensor of token indices
# Batch size = 2, Sequence length = 5
input_indices = torch.randint(0, config.vocab_size, (batch_size, seq_len))

# Get the model's output (logits)
output_logits = model(input_indices)

# Print shapes to verify
print(f"\nInput shape (indices): {input_indices.shape}")
print(f"Output shape (logits): {output_logits.shape}")
print(f"Expected logits shape: {(batch_size, seq_len, config.vocab_size)}")

assert output_logits.shape == (batch_size, seq_len, config.vocab_size)
print("\nSuccess! The config parameters correctly define the model architecture.")

# --- Visualization of the Block ---
print("\n--- Visualizing the Data Flow in a Transformer Block ---")
print("""
Input (B, T, C)
   |
   |-------------------------------------.
   |                                     |
[ LayerNorm ]                            |
   |                                     |
[ Causal Self-Attention ] (discussion)   |
   |                                     |
   '----(+) (add) <----------------------' (residual connection 1)
           |
           |-----------------------------.
           |                             |
        [ LayerNorm ]                    |
           |                             |
        [ MLP (think) ]                  |
           |                             |
           '----(+) (add) <--------------' (residual connection 2)
                   |
            Output (B, T, C)
""")

### Summary

**What We Built:**
We have successfully constructed the entire GPT architecture from the ground up. We started with the core components—Causal Self-Attention and an MLP—and assembled them into a `Block`. We then stacked these blocks, added token and positional embeddings, and attached a final prediction head to create a complete language model.

**Key Takeaways:**
*   **Compositionality:** Complex models like GPT are built from simple, reusable components. The `Block` is the fundamental, repeated unit.
*   **Attention as Communication:** The Causal Self-Attention mechanism is how tokens in a sequence share information with each other, creating context-aware representations. The causal mask is the key for auto-regressive generation.
*   **MLP as Computation:** The feed-forward network is where each token "thinks" about the information it gathered during the attention phase, allowing for more complex computations.
*   **Residuals are Vital:** The `x = x + sublayer(norm(x))` pattern is essential for training deep transformers. It prevents the model from "forgetting" information as data flows through the network.
*   **Config Drives Architecture:** The `GPTConfig` dataclass isn't just for organization; its parameters directly determine the size, shape, and depth of the neural network we instantiate.

**What's Next:**
We have an architecture, a complete neural network skeleton. But a skeleton can't do anything on its own—it needs to be trained on data to learn. In the next notebook, `Preparing and Loading Text Data`, we will take a raw text file (like the complete works of Shakespeare), process it into a format our model can understand, and create data loaders to feed it into the model during training.